# PFA + EPPO Adaptive Learning — Complete POC

Full pipeline: **PFA knowledge tracer** → **EPPO recommender** → **evaluation**

### Architecture
```
RealisticStudent  ->  PFATracker (per-session)  ->  EPPOAgent  ->  action: (concept, Bloom level)
      ^                      ^                         |
      +----------------------+--------- reward <-------+
```

| Section | Description |
|---------|-------------|
| A | Config: 8 CE concepts, 6 Bloom levels |
| B | PFA knowledge tracer with similarity propagation |
| C | RealisticStudent: 3PL IRT + forgetting + concept transfer |
| D | EPPOAgent: actor-critic with competence-ceiling mask |
| E | Reward: ALPN-style mastery gain + fit + diversity |
| F | Rollout buffer + GAE + PPO update |
| G | 8-test PFA validation suite |
| H | Training: PFA warm-up then EPPO |
| I | Policy comparison: EPPO vs random vs greedy |
| J | Demo session (step-by-step trace) |
| K | Diagnostic plots |

**Setup:** attach `pfa_eppo_poc.py` as a Kaggle dataset, GPU T4, run all cells top-to-bottom.  
**Runtime:** ~25-30 min on T4.


In [ ]:
import subprocess
subprocess.run(['pip','install','sentence-transformers','scikit-learn','-q'], check=True)
print('Dependencies ready.')


In [ ]:
import sys, os, shutil, glob

DST = '/kaggle/working/pfa_eppo_poc.py'

print('Files under /kaggle/input:')
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(f'  {os.path.join(root, f)}')

matches = glob.glob('/kaggle/input/**/*.py', recursive=True)
target = None
for m in matches:
    try:
        snippet = open(m).read(600).lower()
        if 'pfatracker' in snippet or 'eppoagent' in snippet or 'realisticstudent' in snippet:
            target = m
            break
    except Exception:
        pass

if target:
    shutil.copy(target, DST)
    print(f'Copied: {target}')
elif os.path.exists(DST):
    print(f'Using existing: {DST}')
else:
    raise FileNotFoundError(
        'pfa_eppo_poc.py not found.\n'
        'Click Add data -> upload pfa_eppo_poc.py -> re-run this cell.'
    )

sys.path = [p for p in sys.path if p != '/kaggle/working']
sys.path.insert(0, '/kaggle/working')
print(f'File size: {os.path.getsize(DST):,} bytes')


In [ ]:
import sys, os, torch, numpy as np, matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['figure.figsize'] = (15, 4)

_wdir = '/kaggle/working'
if _wdir not in sys.path:
    sys.path.insert(0, _wdir)

_src = open(os.path.join(_wdir, 'pfa_eppo_poc.py')).read()
for sym in ['PFATracker','EPPOAgent','RealisticStudent','compute_reward','run_pfa_validation']:
    if sym not in _src:
        raise ImportError(f'Missing symbol: {sym} -- wrong file version uploaded.')

from pfa_eppo_poc import (
    Config, PFATracker, RealisticStudent, EPPOAgent,
    compute_reward, RolloutBuffer, ppo_update,
    run_pfa_validation, train,
    evaluate_policies, run_demo,
    plot_training_curves, plot_policy_comparison, plot_bloom_distribution,
)

cfg = Config()

print(f'PyTorch  : {torch.__version__}')
print(f'GPU      : {torch.cuda.is_available()}  ({cfg.DEVICE})')
print(f'Concepts : {cfg.N_CONCEPTS} -> {cfg.CONCEPTS}')
print(f'Actions  : {cfg.N_ACTIONS}  ({cfg.N_CONCEPTS} x {cfg.BLOOM_LEVELS} Bloom levels)')
print(f'State    : {cfg.STATE_DIM}  ({cfg.N_CONCEPTS} concepts x 4 features)')
print()
print('Config:')
print(f'  WARMUP_EPISODES  = {cfg.WARMUP_EPISODES}')
print(f'  N_EPISODES       = {cfg.N_EPISODES}')
print(f'  MAX_STEPS        = {cfg.MAX_STEPS}')
print(f'  BETA_APR (goal)  = {cfg.BETA_APR}')
print(f'  BLOOM_LEVELS     = {cfg.BLOOM_LEVELS}')
print(f'  MIN_COVERAGE     = {cfg.MIN_COVERAGE}')
print('Import OK.')


## Optional: config overrides

Uncomment any line below before running training to adjust the setup.


In [ ]:
# Faster experiment (less accurate, good for a quick smoke test):
# cfg.WARMUP_EPISODES = 100
# cfg.N_EPISODES      = 600
# cfg.MAX_STEPS       = 25

# More aggressive exploration:
# cfg.ENTROPY_COEF = 0.03

# Stricter mastery goal:
# cfg.BETA_APR = 0.78

print('Active config:')
print(f'  Warmup: {cfg.WARMUP_EPISODES}  |  EPPO episodes: {cfg.N_EPISODES}  |  Max steps/session: {cfg.MAX_STEPS}')
print(f'  Goal APR: {cfg.BETA_APR}  |  Min coverage: {cfg.MIN_COVERAGE}')


## Section B — Build PFA similarity graph

The sentence transformer (`BAAI/bge-base-en-v1.5`) embeds all concepts once and builds a cosine-similarity matrix. This runs once and is shared across all tracker instances during training. Takes ~60s on first run (model download + encode).


In [ ]:
print('Building PFA similarity graph (runs once)...')
dummy_tracker = PFATracker(cfg.CONCEPTS, cfg)
sim_matrix = dummy_tracker.sim_matrix

print(f'\nSimilarity matrix ({cfg.N_CONCEPTS} x {cfg.N_CONCEPTS}):')
header = f'{'':>26}' + ''.join(f'{c[:8]:>10}' for c in cfg.CONCEPTS)
print(header)
for i, c in enumerate(cfg.CONCEPTS):
    row = f'{c:<26}' + ''.join(f'{sim_matrix[i,j]:>10.3f}' for j in range(cfg.N_CONCEPTS))
    print(row)

print('\nTop-3 neighbours per concept:')
for i, c in enumerate(cfg.CONCEPTS):
    sims = sim_matrix[i]
    top3 = [(cfg.CONCEPTS[j], sims[j]) for j in np.argsort(-sims) if j != i][:3]
    print(f'  {c:<28}', ' | '.join(f'{n} ({s:.3f})' for n, s in top3))


## Section G — PFA validation (8-test suite)

Validate the PFA tracker in isolation before attaching EPPO. All 8 tests should pass. If ECE >= 0.15, tune `GAMMA_LEVEL` or `RHO_LEVEL` in Config.

| Test | What it checks | Target |
|------|---------------|--------|
| T1 | Always-correct -> monotone rise | 0 drops, gain > 0.05 |
| T2 | Always-wrong -> monotone drop | 0 rises, drop > 0.05 |
| T3 | Bloom ordering L1 > L2 > ... > L6 | Monotone decreasing |
| T4 | Calibration (ECE) | ECE < 0.10 |
| T5 | Steps to 80% mastery | 10-70 steps |
| T6 | Propagation ordering | Near neighbour moves more |
| T7 | Ghost counts | Raw counts stay 0 for non-attempted |
| T8 | Recovery after failure | Ratio > 70% |


In [ ]:
val_rng = np.random.default_rng(99)
val_results = run_pfa_validation(cfg, sim_matrix, val_rng)

if val_results.get('ece', 1.0) >= 0.15:
    print('\nWARNING: ECE >= 0.15. Consider tuning GAMMA_LEVEL / RHO_LEVEL before EPPO training.')
else:
    print('\nPFA validation looks good -- proceeding to EPPO training.')


## Section H — Training

**Phase 1** (warm-up): 300 random episodes to confirm the PFA similarity graph handles diverse trajectories.

**Phase 2** (EPPO): 1500 episodes of actor-critic training. Each episode samples a fresh PFA tracker and a fresh `RealisticStudent`.

Expected runtime on T4: ~23-28 min.

Watch the log:
- **Steps** should average 12-25 (not 1-3 -- that's the co-training collapse from the old version)
- **Covered** should reach 8/8 most episodes
- **APR** should trend upward, crossing 0.65 by episode 800


In [ ]:
agent, sim_matrix, metrics = train(cfg, save_dir='/kaggle/working/checkpoints')


## Section K — Training curves

What to look for:
- **APR curve**: steady rise toward 0.72, no sudden collapse
- **Reward**: positive trend, values in range [-20, +50]
- **Goal %**: should exceed 50% by episode 700
- **Losses**: actor stabilises, critic decreases


In [ ]:
plot_training_curves(metrics, cfg, '/kaggle/working/training_curves.png')

final_n = -300
print(f'\nFinal {abs(final_n)} episodes summary:')
print(f'  Mean APR   : {np.mean(metrics["aprs"][final_n:]):.3f}')
print(f'  Goal rate  : {np.mean(metrics["goals"][final_n:])*100:.1f}%')
print(f'  Mean steps : {np.mean(metrics["steps"][final_n:]):.1f}')
print(f'  Mean reward: {np.mean(metrics["rewards"][final_n:]):+.2f}')


## Section I — Policy comparison

Compare EPPO against two baselines on 300 fresh students each:

- **Random**: uniform random (concept, Bloom level)
- **Greedy**: always pick weakest concept at its ideal Bloom level (strong hand-crafted heuristic)
- **EPPO**: trained agent

EPPO must beat **Random** to demonstrate it learned something.  
EPPO beating **Greedy** in APR *or* reaching the same APR in fewer steps means it has learned a non-trivial policy.


In [ ]:
policies = evaluate_policies(agent, cfg, sim_matrix, n_students=300)
plot_policy_comparison(policies, cfg, '/kaggle/working/policy_comparison.png')
plot_bloom_distribution(policies, cfg, '/kaggle/working/bloom_dist.png')


## Section J — Demo session

Step-by-step trace of one EPPO session on a fresh student. Shows which concept and Bloom level EPPO chose at each step, whether the student answered correctly, and how APR evolved.

Try `seed=0`, `seed=7`, `seed=99` to see different student profiles.


In [ ]:
run_demo(agent, cfg, sim_matrix, seed=42)


## Readiness checklist

Go/no-go for scaling to the full concept set and connecting to the question generator.


In [ ]:
print('\n' + '='*58)
print('  READINESS CHECKLIST')
print('='*58)

final_200 = slice(-200, None)
mean_apr   = np.mean(metrics['aprs'][final_200])
goal_rate  = np.mean(metrics['goals'][final_200]) * 100
mean_steps = np.mean(metrics['steps'][final_200])
ece        = val_results.get('ece', 1.0)
auc        = val_results.get('auc', 0.0)

eppo_beats_random = policies['EPPO']['mean_apr'] > policies['Random']['mean_apr']
eppo_beats_greedy = (
    policies['EPPO']['mean_apr'] > policies['Greedy']['mean_apr'] or
    policies['EPPO']['mean_steps'] < policies['Greedy']['mean_steps']
)

checks = [
    ('PFA ECE < 0.10',             ece < 0.10,         f'ECE={ece:.4f}'),
    ('PFA AUC > 0.65',             auc > 0.65,         f'AUC={auc:.3f}'),
    ('Final APR > 0.60',           mean_apr > 0.60,    f'APR={mean_apr:.3f}'),
    ('Goal rate > 40%',            goal_rate > 40,     f'{goal_rate:.1f}%'),
    ('Avg steps 10-30',            10 < mean_steps < 30, f'{mean_steps:.1f} steps'),
    ('EPPO beats random',          eppo_beats_random,  f'EPPO {policies["EPPO"]["mean_apr"]:.3f} vs Random {policies["Random"]["mean_apr"]:.3f}'),
    ('EPPO competitive w/ greedy', eppo_beats_greedy,  f'EPPO {policies["EPPO"]["mean_apr"]:.3f} vs Greedy {policies["Greedy"]["mean_apr"]:.3f}'),
]

all_pass = True
for label, ok, detail in checks:
    mark = 'PASS' if ok else 'FAIL'
    print(f'  [{mark}] {label:<35} {detail}')
    if not ok:
        all_pass = False

print()
if all_pass:
    print('  All checks passed -- ready to scale to full concept set.')
else:
    print('  Some checks failed. Suggestions:')
    if ece >= 0.10:
        print('    ECE high: tune GAMMA_LEVEL down or align BETA_LEVEL to student IRT curve.')
    if goal_rate <= 40:
        print('    Goal rate low: increase N_EPISODES or reduce BETA_APR threshold.')
    if mean_steps >= 30:
        print('    Sessions too long: check MIN_COVERAGE or MAX_SAME_CONCEPT.')
    if not eppo_beats_random:
        print('    EPPO not beating random: increase N_EPISODES to 2500 or raise LR_ACTOR to 5e-4.')
print('='*58)
